# Anima / SDXL 生图 + Wan 视频 - Colab 免费档实测版

面向**免费档 T4 16GB / 12.6GB 系统内存 / 102GB 临时盘 / 3.5-6h 会话**。

## 档位自适应

| 档位 | 能做什么 | 本 notebook 行为 |
|---|---|---|
| **GPU 档**（T4 16GB） | Anima 生图 5-10s、SDXL 15-40s、Wan 2.2 5B 视频 | 全功能 |
| **CPU 档**（无卡） | OpenVINO 量化生图，4 档可切（默认 **sana06** / sdxl_lightning / animagine / sd15_lcm）、ffmpeg 后处理 | 自动切 CPU 模型，服务照常出图，只是慢；`cpu_preset` 免重启切档 |

## 修复与新增（相对旧版）

- **fp16 修复**：T4 是 Turing，没有 bf16，`bfloat16` 会掉到无意义的软件模拟。已按卡自动选 `fp16`。
- **会话保活**：免费档约 90 分钟无交互断线，新增 JS 保活。
- **产出落盘**：`/content` 是临时盘（会话结束即清空），新增自动上传 Drive。
- **批量模式**：一次提交 N 条 prompt，整批出图后统一下载。
- **视频支持**：Wan 2.2 TI2V-5B（8GB 级）图生视频，含 ffmpeg 合流。
- **CPU 档可生图（多预设）**：无 GPU 时用 OpenVINO 量化模型，默认 **sana06 = SANA 0.6B**（比 SDXL 更小更快、写实强、2.3GB/4步），另可切 `sdxl_lightning`（通用写实）、`animagine`（动漫 SDXL）、`sd15_lcm`（最小 0.9GB 最快但画质矮）；切档请求带 `cpu_preset=<key>` 或改 cell 4 的 `CPU_PRESET`。
- **时长纠正**：Colab 免费档**不是 12 小时**，实测 3.5-6h，按 3h 规划。

跑完第 4 格打印 `https://xxx.trycloudflare.com`，本机用 `colab_gen.sh "prompt" <url>` 取图。


In [ ]:
#@title 1. 环境探测 + 依赖安装（先跑这格）
import os, sys, subprocess, shutil, platform, time

IN_COLAB = os.path.isdir('/content')
print('colab:', IN_COLAB)

# 内存：12.6GB 是免费档标准系统内存
try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'RAM : {vm.total/1024**3:.1f} GB total, {vm.available/1024**3:.1f} GB free')
except ImportError:
    pass

du = shutil.disk_usage('/content' if IN_COLAB else '.')
print(f'disk: {du.free/1024**3:.1f} GB free')

# GPU 探测
HAS_GPU = False
GPU_NAME, VRAM_GB = 'none', 0.0
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, timeout=30)
    if out.returncode == 0 and out.stdout.strip():
        line = out.stdout.strip().split('\n')[0]
        GPU_NAME = line.split(',')[0].strip()
        VRAM_GB = float(line.split(',')[1].replace('MiB', '').strip()) / 1024
        HAS_GPU = True
except Exception as e:
    print('nvidia-smi failed:', e)

print(f'GPU : {GPU_NAME} ({VRAM_GB:.1f} GB)' if HAS_GPU else 'GPU : 无 -> CPU 档，走 OpenVINO SDXL-Lightning 生图（慢但可用）')

# dtype 选择（按卡能力）
import torch
DTYPE = torch.float16
if HAS_GPU and torch.cuda.get_device_capability(0)[0] >= 8:
    DTYPE = torch.bfloat16
print('dtype:', DTYPE, '| torch', torch.__version__)

# 分步安装依赖，避免一次安装失败
if IN_COLAB:
    print('[*] Installing base packages...')
    %pip -q install diffusers transformers accelerate safetensors flask flask-cors
    %pip -q install psutil pillow requests imageio imageio-ffmpeg

    if HAS_GPU:
        print('[*] GPU 档：生图用 Anima（第 3 格加载）。跑视频前把 INSTALL_DIFFUSERS_MAIN 改 True 再跑本格。')
    else:
        # CPU 档：OpenVINO 跑 SDXL-Lightning INT4，2.06GB 权重 / 4 步 / 无需 CUDA。
        print('[*] CPU 档：装 optimum-intel + openvino（纯 CPU 生图，约 1-2 分钟）...')
        %pip -q install optimum-intel openvino

INSTALL_DIFFUSERS_MAIN = False
if INSTALL_DIFFUSERS_MAIN:
    %pip -q install git+https://github.com/huggingface/diffusers
    print('[+] diffusers 已换主干版（跑 Wan 视频前需要）')
    
print('[+] 基础依赖就绪。如需跑视频，请确保 diffusers 主版本')

In [ ]:
#@title 2. 会话保活 + 产出落盘（防断线 / 防丢产出）
import os, time, shutil, threading

OUT_DIR = '/content/out' if IN_COLAB else './out'
os.makedirs(OUT_DIR, exist_ok=True)

# --- 保活：免费档约 90min 无交互断线。这段 JS 定期点一下 notebook。
KEEPALIVE = True
if IN_COLAB and KEEPALIVE:
    from IPython.display import display, Javascript
    display(Javascript('''
    function keepAlive() {
      const btn = document.querySelector('colab-connect-button');
      if (btn && btn.shadowRoot) {
        const c = btn.shadowRoot.querySelector('#connect');
        if (c) { c.click(); console.log('keepalive ping', new Date().toLocaleTimeString()); }
      }
    }
    setInterval(keepAlive, 60000);
    ''') )
    print('[+] keepalive 已挂载（60s 一次）')

# --- 落盘：/content 会话结束清空，产出必须搬走
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/ai_out'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    DRIVE_OK = True
    print('[+] Drive 已挂载 ->', DRIVE_DIR)
    print('    注意：免费 Drive 仅 15GB，只存 LoRA / 成品 / 数据集')
except Exception as e:
    print('[!] Drive 未挂载（可跳过，用浏览器下载）:', str(e)[:160])

def flush_to_drive(verbose=True):
    """把 out/ 里的产出搬到 Drive，然后清空临时目录防爆盘。"""
    if not DRIVE_OK:
        if verbose: print('[!] Drive 不可用，产出留在', OUT_DIR)
        return
    tgt = os.path.join(DRIVE_DIR, time.strftime('%m%d_%H%M'))
    os.makedirs(tgt, exist_ok=True)
    n = 0
    for root, _, files in os.walk(OUT_DIR):
        for f in files:
            src = os.path.join(root, f)
            shutil.copy2(src, os.path.join(tgt, f)); n += 1
    if verbose: print(f'[+] 已搬 {n} 个文件 -> {tgt}')

print('[+] 输出目录:', OUT_DIR)

In [ ]:
#@title 3. 加载 Anima 模型（GPU 档；CPU 档自动跳过）
# Anima 官方 diffusers 权重，6.8GB，T4 装得下。
MODEL_ID = 'circlestone-labs/Anima-Base-v1.0-Diffusers'
pipe = None

if HAS_GPU:
    from diffusers import ModularPipeline
    pipe = ModularPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
    pipe.load_components()          # 当前 diffusers 的正确 API
    pipe.to('cuda')
    pipe.set_progress_bar_config(disable=True)
    # 省显存：T4 只有 16GB，注意力切成内存高效实现
    try:
        pipe.enable_attention_slicing()
        print('[+] attention slicing on')
    except Exception:
        pass
    print(f'[+] Anima 已加载: {MODEL_ID}')
    print(f'    dtype={DTYPE}  VRAM={torch.cuda.memory_allocated()/1024**3:.1f}GB')
else:
    print('[!] 无 GPU，跳过模型加载。本会话只能做 ffmpeg 后处理/打标。')

In [ ]:
#@title 4. 启动 HTTP 服务 + cloudflared 公网穿透（DSH 直调的入口）
# 修正旧版两个致命问题：
#   1) 旧版给每个任务起一个线程共用同一张 T4 -> 并发推理直接 OOM。现在单 worker 串行。
#   2) 旧版把 PNG bytes 全留在内存 JOBS 里 -> 12.6GB 系统内存装不下大批产出。现在落盘。
import threading, uuid, os, io, json, time, re, queue, subprocess
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

JOBS = {}
LOCK = threading.Lock()
TASK_Q = queue.Queue()

VID_MODEL = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
VIDEO = {'pipe': None, 'loading': False, 'error': None}

# ---- CPU 档生图：OpenVINO INT4 多预设（纯 CPU，无 CUDA）----
# 官方跑分都在 192 vCPU 服务器测；你只有 2 vCPU，按 ~4x 慢算，所以各预设把边长压小。
# 切档：改下面的 CPU_PRESET，或在 /generate / /batch 的 JSON 里传 "cpu_preset": "<key>"（免重启）。
CPU_PRESETS = {
    # 更小的 SDXL 级：SANA 0.6B DiT，权重 2.3GB，peak RSS ~7.2GB，写实最强、最推荐
    'sana06':        {'model': 'HelloSun/Sana_Sprint_0.6B_1024px-OpenVINO-INT4',
                      'steps': 4, 'guidance': 4.5, 'maxdim': 1024, 'neg': False,
                      'extra': {'intermediate_timesteps': None}},
    # SDXL 通用写实：dreamshaper-xl-lightning，2.06GB / 4步 / guidance 2.0
    'sdxl_lightning':{'model': 'HelloSun/dreamshaper-xl-lightning-OpenVINO-INT4',
                      'steps': 4, 'guidance': 2.0, 'maxdim': 768, 'neg': True, 'extra': {}},
    # 动漫 SDXL：animagine-xl-4.0-Lightning。实测体积 6.18GB（unet INT8 2.58 + 其余 FP32 3.61），
    # 不是名字说的 INT4；1024px 4步 guidance0（无 CFG，负向词无效）192vCPU 实测 10.26s/张。
    # 2vCPU 内存吃紧时别用——6.2GB 权重 + 1024px 激活会把 12.6GB 顶到极限。
    'animagine':     {'model': 'HelloSun/animagine-xl-4.0-Lightning-OpenVINO-INT4',
                      'steps': 4, 'guidance': 0.0, 'maxdim': 1024, 'neg': False, 'extra': {}},
    # SDXL 动漫向，体积档位正好落在 ~2.7GB 那一档：nova_xl_lcm INT4（2.06GB 权重，8步 cfg1.5）
    # 比 SDXL-base INT4 快一半（2vCPU 估 8-16 分钟/张 1024px），画质是 SDXL 底子，明显高过 SD1.5
    'sdxl_anime':    {'model': 'HelloSun/nova_xl_lcm-OpenVINO-INT4',
                      'steps': 8, 'guidance': 1.5, 'maxdim': 1024, 'neg': True, 'extra': {}},
    # 最小最快：SD1.5 animemix LCM，实测 0.90GB / 8步 cfg1.5（SD1.5 底子矮，这是画质代价）
    # 192vCPU 实测 1024px 33.87s、512px 5.51s/张；2vCPU 同尺寸要 10-20 倍
    'sd15_lcm':      {'model': 'HelloSun/animemixv10_lcm-OpenVINO-INT4',
                      'steps': 8, 'guidance': 1.5, 'maxdim': 512, 'neg': True, 'extra': {}},
}
CPU_PRESET = 'sana06'
CPU = {'pipe': None, 'key': None, 'loading': False, 'error': None}


def set_cpu_preset(name):
    global CPU_PRESET
    if name in CPU_PRESETS and name != CPU_PRESET:
        CPU_PRESET = name
        CPU['pipe'] = None
        CPU['key'] = None
        CPU['error'] = None


def _avail_gb():
    """当前可用系统内存（GB）；探测失败返回 None（不拦加载）。"""
    try:
        import psutil
        return psutil.virtual_memory().available / 1024 ** 3
    except Exception:
        try:
            with open('/proc/meminfo') as f:
                for ln in f:
                    if ln.startswith('MemAvailable'):
                        return int(ln.split()[1]) / 1024 ** 2
        except Exception:
            pass
        return None


# 各预设权重体积（GB，实测 repo 内 openvino_model.bin 加总），用于加载前的内存闸门
WEIGHT_GB = {'sana06': 2.45, 'sdxl_lightning': 2.06, 'sdxl_anime': 2.06,
             'animagine': 6.18, 'sd15_lcm': 0.90}


def load_cpu_pipe():
    """懒加载当前预设；切档时 key 不符会自动重载。"""
    if CPU['pipe'] is not None and CPU['key'] == CPU_PRESET:
        return CPU['pipe']
    if CPU['loading']:
        raise RuntimeError('CPU model still loading (download + compile), retry in 120s')
    # 换档前先放掉上一份权重：12.6GB 免费档里两份 SDXL 同时驻留必 OOM
    if CPU['pipe'] is not None:
        import gc
        try:
            del CPU['pipe']
        except Exception:
            pass
        CPU['pipe'] = None
        CPU['key'] = None
        gc.collect()
    pr = CPU_PRESETS[CPU_PRESET]
    need = WEIGHT_GB.get(CPU_PRESET, 2.1) + (1.5 if pr['maxdim'] <= 512 else
                                             3.0 if pr['maxdim'] <= 768 else 5.0)
    avail = _avail_gb()
    if avail and avail < need:
        CPU['error'] = ('内存不足：剩 %.1fGB，< %s 预设约需 %.1fGB（权重 %.1f + 激活）。'
                        '换 sd15_lcm / sdxl_lightning，或把 maxdim 调小后重试。'
                        % (avail, CPU_PRESET, need, WEIGHT_GB.get(CPU_PRESET, 2.1)))
        raise RuntimeError(CPU['error'])
    CPU['loading'] = True
    CPU['error'] = None
    try:
        from optimum.intel import OVDiffusionPipeline
        p = OVDiffusionPipeline.from_pretrained(pr['model'], compile=True)
        CPU['pipe'] = p
        CPU['key'] = CPU_PRESET
        print('[+] CPU model ready: %s (%d steps, guidance %.1f, <=%dpx)'
              % (CPU_PRESET, pr['steps'], pr['guidance'], pr['maxdim']))
        return p
    except Exception as e:
        CPU['error'] = 'CPU model load failed: %s: %s' % (type(e).__name__, e)
        raise
    finally:
        CPU['loading'] = False


def set_job(jid, **kw):
    with LOCK:
        JOBS.setdefault(jid, {'status': 'queued', 'created': time.time()}).update(kw)


def get_job(jid):
    with LOCK:
        j = JOBS.get(jid)
        return dict(j) if j else None


def load_video_pipe():
    """视频权重懒加载：只有第一次请求视频时才下载 ~10-20GB。"""
    if VIDEO['pipe'] is not None:
        return VIDEO['pipe']
    if VIDEO['loading']:
        raise RuntimeError('video model still loading, retry in 60s')
    if VIDEO['error']:
        raise RuntimeError(VIDEO['error'])
    VIDEO['loading'] = True
    try:
        import torch
        from diffusers import WanPipeline, AutoencoderKLWan
        vae = AutoencoderKLWan.from_pretrained(VID_MODEL, subfolder='vae', torch_dtype=torch.float32)
        p = WanPipeline.from_pretrained(VID_MODEL, vae=vae, torch_dtype=DTYPE)
        try:
            p.enable_model_cpu_offload()
            p.enable_vae_slicing()
        except Exception:
            p.to('cuda')
        VIDEO['pipe'] = p
        print('[+] video pipeline ready')
        return p
    except Exception as e:
        VIDEO['error'] = 'video model load failed: %s: %s' % (type(e).__name__, e)
        raise RuntimeError(VIDEO['error'])
    finally:
        VIDEO['loading'] = False


def render_image(d):
    p = (d.get('prompt') or '').strip()
    if p and not p.startswith('masterpiece'):
        p = 'masterpiece, best quality, score_7, ' + p
    neg = d.get('negative') or (
        'worst quality, low quality, score_1, score_2, score_3, artist name, '
        'blurry, jpeg artifacts, chromatic aberration')
    import torch
    if HAS_GPU and pipe is not None:
        out = pipe(prompt=p, negative_prompt=neg,
                   height=int(d.get('height') or 1216), width=int(d.get('width') or 832),
                   num_inference_steps=int(d.get('steps') or 25),
                   generator=torch.Generator('cuda').manual_seed(int(d.get('seed') or 0)))
    elif not HAS_GPU:
        if d.get('cpu_preset'):
            set_cpu_preset(str(d['cpu_preset']))
        pr = CPU_PRESETS[CPU_PRESET]
        ov = load_cpu_pipe()
        w = min(int(d.get('width') or pr['maxdim']), pr['maxdim'])
        h = min(int(d.get('height') or pr['maxdim']), pr['maxdim'])
        w -= w % 8
        h -= h % 8
        kw = dict(height=h, width=w, num_inference_steps=pr['steps'],
                  guidance_scale=pr['guidance'],
                  generator=torch.Generator().manual_seed(int(d.get('seed') or 0)))
        kw.update(pr.get('extra') or {})
        if pr.get('neg', True):
            kw['negative_prompt'] = neg
        out = ov(prompt=p, **kw)
    else:
        raise RuntimeError('GPU session but Anima not loaded - run cell 3 first')
    buf = io.BytesIO()
    out.images[0].save(buf, format='PNG')
    return buf.getvalue(), 'png'


def render_video(d):
    from diffusers.utils import export_to_video, load_image
    import torch
    vp = load_video_pipe()
    VID_NEG = d.get('negative') or (
        '色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，'
        '最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，'
        '画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，'
        '杂乱的背景，三条腿，背景人很多，倒着走')
    kw = dict(prompt=d.get('prompt') or '', negative_prompt=VID_NEG,
              height=int(d.get('height') or 704), width=int(d.get('width') or 1280),
              num_frames=int(d.get('frames') or 121), guidance_scale=float(d.get('guidance') or 5.0),
              num_inference_steps=int(d.get('steps') or 50),
              generator=torch.Generator('cuda').manual_seed(int(d.get('seed') or 0)))
    src = d.get('image_path') or ''
    mode = 't2v'
    if src and os.path.exists(src):
        kw['image'] = load_image(src)
        mode = 'i2v'
    frames = vp(**kw).frames[0]
    raw = os.path.join(OUT_DIR, 'v_%d_raw.mp4' % int(time.time()))
    export_to_video(frames, raw, fps=int(d.get('fps') or 24))
    final = raw.replace('_raw.mp4', '.mp4')
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', raw,
                    '-c:v', 'libx264', '-crf', '23', '-preset', 'fast',
                    '-movflags', '+faststart', final], check=True)
    try:
        os.remove(raw)
    except OSError:
        pass
    return open(final, 'rb').read(), 'mp4'


def worker():
    """单线程消费队列：同一时刻只有一个推理任务，避免并发 OOM。"""
    while True:
        jid, kind, d = TASK_Q.get()
        set_job(jid, status='running', kind=kind)
        t0 = time.time()
        try:
            if kind == 'video' and not HAS_GPU:
                raise RuntimeError('video needs GPU (Wan2.2); CPU 档只能生图')
            blob, ext = (render_video(d) if kind == 'video' else render_image(d))
            path = os.path.join(OUT_DIR, '%s.%s' % (jid, ext))
            open(path, 'wb').write(blob)
            set_job(jid, status='done', seconds=round(time.time() - t0, 1),
                    path=path, bytes=len(blob), ext=ext)
            print('[+] %s %s done in %.1fs' % (kind, jid[:8], time.time() - t0))
        except Exception as e:
            set_job(jid, status='error', error=('%s: %s' % (type(e).__name__, e))[:400])
        finally:
            TASK_Q.task_done()


def submit(kind, d):
    jid = uuid.uuid4().hex
    set_job(jid, status='queued', kind=kind)
    TASK_Q.put((jid, kind, d))
    return jid


@app.route('/cpu_presets')
def cpu_presets_ep():
    return jsonify(current=CPU_PRESET, presets={k: v['model'] for k, v in CPU_PRESETS.items()})


@app.route('/health')
def health():
    return jsonify(ok=True, gpu=HAS_GPU, gpu_name=GPU_NAME, vram_gb=round(VRAM_GB, 1),
                   pending=TASK_Q.qsize(), jobs=len(JOBS), out=OUT_DIR,
                   video_loaded=VIDEO['pipe'] is not None, video_error=VIDEO['error'],
                   cpu_ready=CPU['pipe'] is not None, cpu_error=CPU['error'],
                   cpu_preset=CPU_PRESET, cpu_presets=list(CPU_PRESETS),
                   cpu_weight_gb=WEIGHT_GB.get(CPU_PRESET),
                   ram_free_gb=round(_avail_gb() or 0, 1))


@app.route('/generate', methods=['POST'])
def generate():
    d = request.get_json(force=True)
    if not (d.get('prompt') or '').strip():
        return jsonify(error='prompt required'), 400
    kind = d.get('kind') or 'image'
    if kind not in ('image', 'video'):
        return jsonify(error='kind must be image|video'), 400
    return jsonify(id=submit(kind, d), queued=TASK_Q.qsize())


@app.route('/batch', methods=['POST'])
def batch():
    d = request.get_json(force=True)
    prompts = d.get('prompts') or []
    if not prompts:
        return jsonify(error='prompts required'), 400
    n = int(d.get('n_per_prompt') or 1)
    kind = d.get('kind') or 'image'
    ids = []
    for pr in prompts:
        for k in range(n):
            dd = dict(d)
            dd['prompt'] = pr
            if d.get('seed'):
                dd['seed'] = int(d['seed']) + k
            ids.append(submit(kind, dd))
    return jsonify(ids=ids, count=len(ids), queued=TASK_Q.qsize())


@app.route('/status/<jid>')
def status(jid):
    j = get_job(jid)
    if not j:
        return jsonify(error='unknown id'), 404
    j.pop('png', None)
    return jsonify(j)


@app.route('/image/<jid>')
def image(jid):
    j = get_job(jid)
    if not j or j.get('status') != 'done':
        return jsonify(error='not ready'), 404
    return send_file(j['path'])


@app.route('/result/<jid>')
def result(jid):
    """通用取结果（图片/视频都走这里，带正确 mimetype）。"""
    j = get_job(jid)
    if not j or j.get('status') != 'done':
        return jsonify(error='not ready'), 404
    mime = 'video/mp4' if j.get('ext') == 'mp4' else 'image/png'
    return send_file(j['path'], mimetype=mime)


@app.route('/upload', methods=['POST'])
def upload():
    """上传参考静帧给 I2V 用（DSH 端把本地图推上来）。"""
    f = request.files.get('file')
    if not f:
        return jsonify(error='file field required'), 400
    name = re.sub(r'[^A-Za-z0-9_.-]', '_', f.filename or 'ref.png')
    path = os.path.join(OUT_DIR, 'up_' + name)
    f.save(path)
    return jsonify(ok=True, path=path)


@app.route('/flush')
def flush():
    flush_to_drive()
    return jsonify(ok=True, drive=DRIVE_OK)


threading.Thread(target=worker, daemon=True).start()
threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=7860, debug=False, use_reloader=False),
    daemon=True).start()

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                    '-O', '/usr/local/bin/cloudflared'])
    os.chmod('/usr/local/bin/cloudflared', 0o755)

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for line in proc.stdout:
    if 'trycloudflare.com' in line:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            URL = m.group(0)
            break

print('=' * 64)
print('PUBLIC URL:', URL)
print('=' * 64)
if not HAS_GPU:
    print('[i] CPU 档当前预设 %s（模型=%s）；切档改 CPU_PRESET 或请求带 cpu_preset=sana06/sdxl_lightning/animagine/sd15_lcm' % (CPU_PRESET, CPU_PRESETS[CPU_PRESET]['model']))
print('存到本机（DSH 插件会自动读这个文件）:')
print("  echo '%s' > ~/.dsh/.colab-url" % URL)
print('本机 CLI:')
print('  scripts/colab_gen.sh "prompt"')


In [ ]:
#@title 5. 自测：出一张 + 验证全链路（GPU / CPU 两档都测）
if not HAS_GPU:
    print('[i] CPU 档：首张会先下 2GB 权重 + 编译（OpenVINO），可能要 5-10 分钟，耐心等。')
import requests, time, io
r = requests.post(URL + '/generate', json={
    'prompt': '1girl, solo, blue long hair, sitting on bed, looking at viewer, white t-shirt, soft morning window light, volumetric lighting, detailed skin texture',
    'width': 832, 'height': 1216, 'steps': 25, 'seed': 12345,
})
jid = r.json()['id']
print('job', jid)
deadline = time.time() + (900 if HAS_GPU else 1500)   # GPU 15min / CPU 25min 兜底
s = {}
while time.time() < deadline:
    s = requests.get(URL + '/status/' + jid, timeout=30).json()
    print(' ', s.get('status'), s.get('seconds', ''), s.get('error', ''))
    if s.get('status') in ('done', 'error'):
        break
    time.sleep(5)
if s.get('status') == 'done':
    from PIL import Image
    img = requests.get(URL + '/image/' + jid, timeout=120).content
    im = Image.open(io.BytesIO(img))
    print('  出图成功:', im.size, f'{len(img)/1024:.0f} KB, {s.get("seconds")}s')
    display(im)
    print('  GPU(T4) Anima 25 步约 15-40s；CPU(OpenVINO 4步, 边长压到768) 约 3-10 分钟，都属正常')
else:
    print('  [!] 未完成:', s.get('error') or 'timeout - CPU 首张在下载/编译，可重跑本格再等')

In [ ]:
#@title 6. 批量出图（GPU / CPU 两档都可用）
# 每行一条 prompt。GPU 建议 20-40 张/批；CPU 每张要 3-10 分钟，建议每批 <=5 张。
PROMPTS = [
    '1girl, solo, silver hair, standing in rain, wet clothes, city night, neon reflections',
    '1girl, solo, red kimono, cherry blossoms, sunlight through petals, bokeh',
    '1girl, solo, black hair, reading book in library, warm lamp light, dust motes',
]
N_PER_PROMPT = 2
BATCH_STEPS  = 22          # 22-25 步质量/速度平衡点；CPU 档会被忽略（Lightning 固定 4 步）
W, H         = 832, 1216   # CPU 档按当前预设的 maxdim 自动压边长（sana06=1024/sd15_lcm=512）
SEED_BASE    = 1000

if 'URL' not in globals() or not URL:
    print('[!] 先跑第 4 格起服务。')
else:
    import requests, time
    if not HAS_GPU:
        print('[i] CPU 档：steps 被忽略（固定 4 步），每张较慢，建议先少量试跑。')
    r = requests.post(URL + '/batch', json={
        'prompts': PROMPTS, 'n_per_prompt': N_PER_PROMPT,
        'width': W, 'height': H, 'steps': BATCH_STEPS, 'seed': SEED_BASE,
    })
    ids = r.json().get('ids', [])
    per = 30 if HAS_GPU else 360          # 单张估算秒数
    print(f'[+] 已入队 {len(ids)} 张，预计约 {len(ids) * per / 60:.0f} 分钟')
    t0 = time.time(); done = 0
    deadline = t0 + len(ids) * per * 4 + 600      # 4 倍余量兜底
    while time.time() < deadline:
        st = [get_job(i) or {'status': '?'} for i in ids]
        d = sum(1 for s in st if s['status'] == 'done')
        e = sum(1 for s in st if s['status'] == 'error')
        if d != done:
            done = d
            el = (time.time() - t0) / 60
            left = (el / max(d, 1)) * (len(ids) - d) if d else 0
            print(f'  {d}/{len(ids)}  用时 {el:.1f}min  剩余约 {left:.0f}min')
        if d + e >= len(ids):
            break
        time.sleep(5)
    st = [get_job(i) or {'status': '?'} for i in ids]
    ok = sum(1 for s in st if s['status'] == 'done')
    err = sum(1 for s in st if s['status'] == 'error')
    print(f'[+] 完成 {ok} 失败 {err}，总用时 {(time.time()-t0)/60:.1f} 分钟')
    flush_to_drive()
    print('[!] 会话结束 /content 会清空，确认上面已搬进 Drive')

In [ ]:
#@title 7. 视频：Wan 2.2 TI2V-5B（走第 4 格的队列，单份模型不 OOM）
# 关键：视频权重【懒加载】——只有你第一次请求视频时，服务端才下载 ~10-20GB。
# 只跑图的话永远不会下视频模型，不占带宽也不占显存。
MAKE_VIDEO = False          # 改成 True 才触发下载+生成
SRC_IMAGE  = ''             # 图生视频：填 cell 4 OUT_DIR 里的静帧路径；留空=文生视频
VID_PROMPT = 'gentle camera push in, hair swaying, soft light shift, subtle breathing motion'
VID_H, VID_W = 704, 1280    # 官方 720P 档
VID_FRAMES   = 121          # 121 帧 @24fps = 5 秒；降到 49 省一半时间
VID_STEPS    = 50           # 官方 50 步；降到 30 提速
VID_FPS      = 24           # TI2V-5B 原生 24fps

if not MAKE_VIDEO:
    print('[i] MAKE_VIDEO=False，跳过。只跑图不会下载视频模型（省 ~10-20GB）。')
elif not HAS_GPU:
    print('[!] 无 GPU，跳过视频。')
elif VRAM_GB < 15:
    print(f'[!] 显存 {VRAM_GB:.1f}GB 偏小；720P 建议 >=16GB（T4 需 offload，勉强）')
elif 'URL' not in globals() or not URL:
    print('[!] 先跑第 4 格起服务。')
else:
    import requests, time, os
    # 走 HTTP 队列：服务端 load_video_pipe() 保证只有一份视频模型
    payload = {'kind': 'video', 'prompt': VID_PROMPT, 'height': VID_H, 'width': VID_W,
               'frames': VID_FRAMES, 'steps': VID_STEPS, 'fps': VID_FPS, 'seed': 42}
    if SRC_IMAGE and os.path.exists(SRC_IMAGE):
        # 有本地参考图：先上传拿服务端路径，再带 image_path 提交（I2V）
        up = requests.post(URL + '/upload', files={'file': open(SRC_IMAGE, 'rb')}, timeout=180)
        payload['image_path'] = up.json()['path']
        print('[*] 模式：图生视频 (I2V)')
    else:
        print('[*] 模式：文生视频 (T2V)')

    r = requests.post(URL + '/generate', json=payload, timeout=60)
    jid = r.json()['id']
    print(f'[*] 已入队 {jid[:8]}（首次会先下 10-20GB 权重，请耐心）')
    t0 = time.time()
    last = ''
    while True:
        s = requests.get(URL + '/status/' + jid, timeout=30).json()
        st = s.get('status')
        if st != last:
            print(f'  {st}  {time.time()-t0:.0f}s  {s.get("error","")}')
            last = st
        if st in ('done', 'error'):
            break
        if time.time() - t0 > 1500:      # 下载+生成 25 分钟兜底
            print('[!] 超时，去 Colab 看服务端日志')
            break
        time.sleep(10)

    if s.get('status') == 'done':
        mp4 = requests.get(URL + '/result/' + jid, timeout=300).content
        out = os.path.join(OUT_DIR, 'vid_%s.mp4' % jid[:8])
        open(out, 'wb').write(mp4)
        print(f'[+] {out}  {len(mp4)/1024**2:.1f} MB  生成 {s.get("seconds")}s')
        flush_to_drive()
    else:
        print('[x] 失败:', s.get('error', 'timeout'))


In [ ]:
#@title 8. 后处理 / 打标 / 编解码（任何档位可用）
# CPU 档现在也能生图（OpenVINO SDXL-Lightning，见第 4/5 格）；想在等图时抢时间可在这里做 ffmpeg / 打标。
import os, glob, subprocess, shutil

MODE = 'probe'      # probe | make_frames | to_h264 | tag | dataset
SRC_DIR = OUT_DIR
print(f'MODE={MODE}  SRC_DIR={SRC_DIR}')
print('ffmpeg:', subprocess.run(['ffmpeg','-version'], capture_output=True, text=True).stdout.split('\n')[0])

def sh(cmd):
    print('$', ' '.join(cmd)[:140])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('  [!]', (r.stderr or '')[-400:])
    return r

if MODE == 'probe':
    # 视频硬件/软件编码器可用性 —— 决定能不能快速编码
    enc = subprocess.run(['ffmpeg','-hide_banner','-encoders'], capture_output=True, text=True).stdout
    for k in ['libx264', 'libx265', 'h264_nvenc', 'av1_mediacodec']:
        print(f'  {k:18s}', 'OK' if k in enc else 'missing')
    if vm:
        print(f'  RAM 可用 {vm.available/1024**3:.1f} GB')

elif MODE == 'make_frames':
    # 视频抽帧 -> 做 LoRA 数据集 / 提取关键帧
    os.makedirs('frames', exist_ok=True)
    for v in glob.glob(os.path.join(SRC_DIR, '*.mp4')):
        sh(['ffmpeg','-y','-i',v,'-vf','fps=2','frames/%04d.png'])
    print('[+] frames/ 共', len(glob.glob('frames/*.png')), '张')

elif MODE == 'to_h264':
    # 云端出的片子可能是 H.265/ProRes，转成手机认得的 H.264
    for v in glob.glob(os.path.join(SRC_DIR, '*.mp4')):
        out = v.replace('.mp4', '_h264.mp4')
        sh(['ffmpeg','-y','-i',v,'-c:v','libx264','-crf','24','-preset','veryfast','-c:a','aac',out])

elif MODE == 'tag':
    # WD14 打标，为 LoRA 训练备数据集（CPU 也能跑，慢但够用）
    try:
        %pip -q install onnxruntime huggingface_hub pandas
        from huggingface_hub import hf_hub_download
        print('[i] WD14 打标：建议在 GPU 档跑，CPU 每小时约 300-600 张')
    except Exception as e:
        print('[!]', e)

elif MODE == 'dataset':
    # 数据集整理：统一尺寸 + 去重 + 重命名（LoRA 训练前必备）
    os.makedirs('dataset', exist_ok=True)
    from PIL import Image
    seen = set(); n = 0
    for p in sorted(glob.glob(os.path.join(SRC_DIR, '*.png'))):
        try:
            im = Image.open(p).convert('RGB')
            k = (im.size, len(im.tobytes()) % 9973)
            if k in seen:
                continue
            seen.add(k)
            side = min(im.size)
            im = im.resize((1024, int(im.height * 1024 / im.width)), Image.LANCZOS) \
                if side >= 1024 else im
            im.save(f'dataset/img_{n:05d}.png')
            n += 1
        except Exception as e:
            print('  skip', p, e)
    print(f'[+] dataset/ 共 {n} 张（去重后）')
    flush_to_drive()

## 实战注意（免费档踩坑清单）

### 会话与档位

- **时长是 3.5-6h，不是 12h**。旧版写的 12 小时是错的，别按 12h 排活；至少留 20 分钟缓冲落盘。
- **可能连到 CPU 档**。CPU 档也能生图（OpenVINO 量化，默认 sana06，见第 4 格预设表），只是 2 vCPU 慢，每预设自带 maxdim 压边长；`cpu_preset` 可切档（sd15_lcm 最快）。抢到 GPU 则走 Anima 更快。等图时可跑第 8 格做后处理。
- **空闲 90 分钟会断线**。第 2 格的 keepalive 已挂，但别关浏览器标签。

### 内存

- **12.6GB 是免费档标准系统内存**，不是缩水。网上说的 25.2GB 是把 15GB 显存算进去了。
- 加载 Flux / 14B 视频模型会 OOM。免费档的天花板就是 **Anima + SDXL + Wan 2.2 5B**。

### 磁盘 102GB

- **`/content` 是临时盘，会话结束即清空**。每次会话都要重下模型（带宽快，5-10 分钟可接受）。
- **别指望复用**。要复用就上 Drive，但免费 Drive 只有 15GB。
- 视频最吃盘：一条 480p/49 帧原生 mp4 约 10-40MB，批量前先看 `du -sh /content`。

### 省钱省时的三个开关

| 开关 | 效果 |
|---|---|
| `steps` 25 → 20 | 快 20%，质量损失肉眼难辨 |
| 先出图再 I2V | 比 T2V 省 3-5 倍 |
| 分辨率 `832x1216` 而非 `1024x1536` | T4 上快近一倍 |

### 相关

- 本机客户端：[scripts/colab_gen.sh](<../scripts/colab_gen.sh>)
- 免费路线全景（AI Horde / Pollinations / ZeroGPU 配额机制）：[docs/routes.md](<../docs/routes.md>)

### 视频的两条硬约束（官方 model card 确认）

- **必须装 diffusers 主干版**：`pip install git+https://github.com/huggingface/diffusers`，PyPI 稳定版还不支持 TI2V-5B。
- **官方参数是 720P / 24fps / 5 秒**（`704x1280`, `num_frames=121`），单卡消费级 GPU 也就 **9 分钟内**。别自己降到 480p/16fps，那是旧版凭感觉写的。
- 官方用 `WanPipeline`（不是 `WanImageToVideoPipeline`），同一个 pipeline 靠是否有 `image=` 参数决定走 T2V 还是 I2V。
